In [ ]:
import os
from experiments.common import paths


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from fff.evaluate.fid import compute_fid_openai_tf as compute_fid
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from tqdm.auto import tqdm
import os
import matplotlib as mpl

mpl.rcParams["mathtext.fontset"] = "stix"
plt.rcParams.update({'font.size': 14})

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class TemperatureScaler(nn.Module):
    """
    Applies temperature scaling to logits:
        scaled_logits = logits / T
    """
    def __init__(self, min_temp=0.5, max_temp=2):
        super().__init__()
        # Initialize T = 1
        self.min_temp = min_temp
        self.max_temp = max_temp
        self.log_temperature = nn.Parameter(torch.zeros(5, device=device))

    @property
    def temperature(self):
        return torch.clamp(torch.exp(self.log_temperature), min=self.min_temp, max=self.max_temp)

    def forward(self, logits):
        return logits / self.temperature


@torch.no_grad()
def collect_logits_and_labels(model, dataloader, device):
    """
    Run model over dataloader and collect logits + labels.
    """
    all_logits = []
    all_labels = []
    model.eval()

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())

    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)


def fit_temperature_scaling(logits, labels, max_iter=200, lr=0.02):
    """
    Fits temperature T on validation logits using binary cross-entropy loss
    for multi-label classification.
    """
    scaler = TemperatureScaler()
    optimizer = optim.LBFGS([scaler.log_temperature], lr=lr, max_iter=max_iter)

    bce = nn.BCEWithLogitsLoss()

    def _eval():
        optimizer.zero_grad()
        scaled_logits = scaler(logits)
        loss = bce(scaled_logits, labels)
        loss.backward()
        return loss

    optimizer.step(_eval)
    return scaler


@torch.no_grad()
def compute_calibration(logits, labels):
    """
    Full wrapper:
    1. Collect logits + labels from dataloader
    2. Fit temperature scaling
    3. Return calibrated probabilities + temperature
    """
    # ---- Step 2: fit calibrator ----
    scaler = fit_temperature_scaling(logits, labels, max_iter=50)

    # ---- Step 3: compute calibrated probabilities ----
    calibrated_logits = scaler(logits)

    return calibrated_logits, scaler

@torch.no_grad()
def tune_threshold(logits, labels, metric="f1", num_thresholds=200):
    """
    logits: [N, C] unnormalized model outputs
    labels: [N, C] ground truth (0/1)
    metric: "f1", "precision", or "recall"
    """
    probs = torch.sigmoid(logits)

    # Candidate thresholds
    thresholds = torch.linspace(0, 1, num_thresholds)

    best_t = - torch.ones(5, device=device)
    best_score = - torch.ones(5, device=device)

    for t in thresholds:
        preds = (probs >= t).float()

        tp = (preds * labels).sum(dim=0)
        fp = ((preds == 1) & (labels == 0)).sum(dim=0)
        fn = ((preds == 0) & (labels == 1)).sum(dim=0)

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        # Mean over classes
        if metric == "precision":
            score = precision
        elif metric == "recall":
            score = recall
        else:
            score = f1
        new_best_ind = score > best_score
        best_t[new_best_ind] = t
        best_score[new_best_ind] = score[new_best_ind]

    return best_t, best_score


In [ ]:
target_columns = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
all_pathologies = [
            "No Finding",
            "Enlarged Cardiomediastinum",
            "Cardiomegaly",
            "Lung Opacity",
            "Lung Lesion",
            "Edema",
            "Consolidation",
            "Pneumonia",
            "Atelectasis",
            "Pneumothorax",
            "Pleural Effusion",
            "Pleural Other",
            "Fracture",
            "Support Devices",
        ]

target_indices = [all_pathologies.index(target) for target in target_columns]

In [ ]:
@torch.no_grad()
def compute_ROC_curve(logits, targets, num_bins=10):
    fpr = []
    tpr = []
    reg_fpr = (~targets).sum() == 0
    reg_tpr = (targets).sum() == 0
    
    for threshold in torch.linspace(1, 0, num_bins):
        quantile = logits.quantile(threshold.to(logits.device))
        labels = logits > quantile      
        fpr.append((labels[~targets].sum()+reg_fpr)/((~targets).sum()+reg_fpr))
        tpr.append((labels[targets].sum()+reg_tpr)/(targets.sum()+reg_tpr))
    return torch.stack(fpr, dim=0), torch.stack(tpr, dim=0)

In [ ]:
@torch.no_grad()
def find_nearest_neighbor_batched_mm(
    image_embeddings: torch.Tensor,
    dataset: torch.Tensor,
    subject_model: torch.nn.Module,
    identity_threshold: float = 1e-4,
    batch_size: int = 1024,
):
    """
    Memory-efficient nearest neighbor search using matrix multiplication.

    Args:
        image_embeddings:    (N_img, D)
        original_embeddings: (N_orig, D)
        identity_threshold:  distances below this are ignored
        batch_size:          number of originals per chunk

    Returns:
        indices: (N_img,) nearest original index per image
    """
    device = image_embeddings.device
    N_img, D = image_embeddings.shape
    N_orig = len(dataset)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    

    # Precompute norms
    img_norms = (image_embeddings ** 2).mean(dim=1)        # (N_img,)
    best_dist = torch.full((N_img,), torch.inf, device=device)
    best_idx = torch.full((N_img,), -1, dtype=torch.long, device=device)

    for n_batch, batch in enumerate(tqdm(dataloader)):
        orig_chunk = subject_model(batch[0].to(device))  # (B, D)

        # (B,)
        orig_norms = (orig_chunk ** 2).mean(dim=1)

        # (B, N_img)
        cross = orig_chunk @ image_embeddings.T             # dot products
        dist = (
            orig_norms[:, None]
            + img_norms[None, :]
            - 2 * cross / D
        )

        # Ignore identical embeddings
        dist[dist < identity_threshold] = torch.inf

        # Best match in this chunk
        chunk_min_dist, chunk_min_idx = torch.min(dist, dim=0)

        # Update global best
        update = chunk_min_dist < best_dist
        best_dist[update] = chunk_min_dist[update]
        best_idx[update] = chunk_min_idx[update] + n_batch*batch_size

    return best_idx

def masked_mean(x, mask, dim):
    mask = mask.float().to(x.device)
    return (x * mask).sum(dim) / mask.sum(dim).clamp(min=1)

In [ ]:

def append_and_save_chexpert_samples(prefix_filter="sampled_invariances_", save_name="appended_chexpert_invariances.pt"):
    combined_dict = {
        "invariances_convnext": [],
        "invariances_biomed": [],
        "originals": [],
        "labels": [],
        "invariances_convnext_embeddings": [],
        "invariances_biomed_embeddings": [],
        "original_convnext_embeddings": [],
        "original_biomed_embeddings": [],
        "invariances_convnext_cross_embeddings": [],
        "invariances_biomed_cross_embeddings": [],
    }
    
    for fname in tqdm(os.listdir(".")):
        if fname.startswith(prefix_filter):
            results_dict = torch.load(fname)
            for key in results_dict.keys():
                combined_dict[key].append(results_dict[key])
    for key in combined_dict.keys():
        combined_dict[key] = torch.cat(combined_dict[key], dim=0)
    torch.save(combined_dict, save_name)

In [ ]:

def combine_and_save_chexpert_samples(prefix_filter="sampled_invariances_", save_name="sampled_chexpert_invariances.pt"):
    combined_dict = {
        "invariances_convnext": [],
        "invariances_biomed": [],
        "originals": [],
        "labels": [],
        "invariances_convnext_embeddings": [],
        "invariances_biomed_embeddings": [],
        "original_convnext_embeddings": [],
        "original_biomed_embeddings": [],
        "invariances_convnext_cross_embeddings": [],
        "invariances_biomed_cross_embeddings": [],
    }
    
    for fname in tqdm(os.listdir(".")):
        if fname.startswith(prefix_filter):
            results_dict = torch.load(fname)
            for key in results_dict.keys():
                combined_dict[key].append(results_dict[key])
    
    # Check consistency:
    for last_orig, next_orig in zip(combined_dict["originals"][:-1], combined_dict["originals"][1:]):
        min_len = min(len(last_orig), len(next_orig))
        assert torch.all(((last_orig[:min_len] - next_orig[:min_len])**2).sum(dim=-1) < 1.e-4), "Dataset order does not agree"
    
    max_len, ind_max = 0, None
    for i, orig in enumerate(combined_dict["originals"]):
        if len(orig) > max_len:
            ind_max = i
            max_len = len(orig)
    
    combined_dict["originals"] = combined_dict["originals"][ind_max]
    combined_dict["original_convnext_embeddings"] = combined_dict["original_convnext_embeddings"][ind_max]
    combined_dict["original_biomed_embeddings"] = combined_dict["original_biomed_embeddings"][ind_max]
    combined_dict["labels"] = combined_dict["labels"][ind_max].to(device)
    
    assert len(combined_dict["originals"]) == len(combined_dict["original_convnext_embeddings"]) == len(combined_dict["original_biomed_embeddings"])
    assert len(combined_dict["originals"]) == len(combined_dict["labels"])
    
    combined_dict["masks"] = []
    for i, (inv_convnext, 
            inv_convnext_embeddings, 
            inv_convnext_cross_embeddings, 
            inv_biomed, 
            inv_biomed_embeddings, 
            inv_biomed_cross_embeddings) in enumerate(zip(combined_dict["invariances_convnext"], 
                                                           combined_dict["invariances_convnext_embeddings"],
                                                           combined_dict["invariances_convnext_cross_embeddings"],
                                                           combined_dict["invariances_biomed"],
                                                           combined_dict["invariances_biomed_embeddings"],
                                                           combined_dict["invariances_biomed_cross_embeddings"])):
        
        assert len(inv_convnext) == len(inv_convnext_embeddings) == len(inv_biomed) == len(inv_biomed_embeddings) == len(inv_convnext_cross_embeddings) == len(inv_biomed_cross_embeddings)
        mask = torch.cat((torch.ones(len(inv_convnext), device=inv_convnext.device, dtype=bool), 
                          torch.zeros(max_len - len(inv_convnext), device=inv_convnext.device, dtype=bool)), dim=0)
        inv_convnext = torch.cat((inv_convnext, torch.zeros(max_len - len(inv_convnext), *inv_convnext.shape[1:], device=inv_convnext.device, dtype=inv_convnext.dtype)), dim=0)
        inv_convnext_embeddings = torch.cat((inv_convnext_embeddings, torch.zeros(max_len - len(inv_convnext_embeddings), *inv_convnext_embeddings.shape[1:], device=inv_convnext_embeddings.device, dtype=inv_convnext_embeddings.dtype)), dim=0)
        inv_convnext_cross_embeddings = torch.cat((inv_convnext_cross_embeddings, torch.zeros(max_len - len(inv_convnext_cross_embeddings), *inv_convnext_cross_embeddings.shape[1:], device=inv_convnext_cross_embeddings.device, dtype=inv_convnext_cross_embeddings.dtype)), dim=0)
        inv_biomed = torch.cat((inv_biomed, torch.zeros(max_len - len(inv_biomed), *inv_biomed.shape[1:], device=inv_biomed.device, dtype=inv_biomed.dtype)), dim=0)
        inv_biomed_embeddings = torch.cat((inv_biomed_embeddings, torch.zeros(max_len - len(inv_biomed_embeddings), *inv_biomed_embeddings.shape[1:], device=inv_biomed_embeddings.device, dtype=inv_biomed_embeddings.dtype)), dim=0)
        inv_biomed_cross_embeddings = torch.cat((inv_biomed_cross_embeddings, torch.zeros(max_len - len(inv_biomed_cross_embeddings), *inv_biomed_cross_embeddings.shape[1:], device=inv_biomed_cross_embeddings.device, dtype=inv_biomed_cross_embeddings.dtype)), dim=0)
        combined_dict["invariances_convnext"][i] = inv_convnext
        combined_dict["invariances_convnext_embeddings"][i] = inv_convnext_embeddings
        combined_dict["invariances_convnext_cross_embeddings"][i] = inv_convnext_cross_embeddings
        combined_dict["invariances_biomed"][i] = inv_biomed
        combined_dict["invariances_biomed_embeddings"][i] = inv_biomed_embeddings
        combined_dict["invariances_biomed_cross_embeddings"][i] = inv_biomed_cross_embeddings
        combined_dict["masks"].append(mask)
    
    combined_dict["invariances_convnext"] = torch.stack(combined_dict["invariances_convnext"], dim=1)
    combined_dict["invariances_convnext_embeddings"] = torch.stack(combined_dict["invariances_convnext_embeddings"], dim=1)
    combined_dict["invariances_convnext_cross_embeddings"] = torch.stack(combined_dict["invariances_convnext_cross_embeddings"], dim=1)
    combined_dict["invariances_biomed"] = torch.stack(combined_dict["invariances_biomed"], dim=1)
    combined_dict["invariances_biomed_embeddings"] = torch.stack(combined_dict["invariances_biomed_embeddings"], dim=1)
    combined_dict["invariances_biomed_cross_embeddings"] = torch.stack(combined_dict["invariances_biomed_cross_embeddings"], dim=1)
    combined_dict["masks"] = torch.stack(combined_dict["masks"], dim=1)
    
    torch.save(combined_dict, save_name)

In [ ]:
# The two audited classifiers, defined once in the experiments package.
from experiments.chexpert.subject_models import (
    BiomedClipSubjectModel,
    ConvNextClassfierSubjectModel,
    renormalize_grayscale,
)

BIOMED_PATH = paths.data("biomedclip-pretrained-larger-chexpert_384")
CONVNEXT_PATH = paths.data("convnextv2-tiny-chexpert_384")


# Sample visualizations

In [ ]:
os.chdir(paths.output("chexpert", "sampled_invariances", "large_biomedclip/v7"))

In [ ]:
combine_and_save_chexpert_samples()

In [ ]:
inv_dict = torch.load("sampled_chexpert_invariances.pt", map_location="cpu")

In [ ]:
orig, mask = inv_dict["originals"], inv_dict["masks"]
inv_biomed, inv_convnext = inv_dict["invariances_biomed"], inv_dict["invariances_convnext"]

In [ ]:
subject_model_biomed = BiomedClipSubjectModel(BIOMED_PATH).to(device)
subject_model_convnext = ConvNextClassfierSubjectModel(CONVNEXT_PATH).to(device)

with torch.no_grad():
    assert ((inv_dict["original_biomed_embeddings"][:100].to(device) - subject_model_biomed(orig[:100].to(device)))**2).mean() < 1.e-4
    assert ((inv_dict["original_convnext_embeddings"][:100].to(device) - subject_model_convnext(orig[:100].to(device)))**2).mean() < 1.e-4

In [ ]:
from fff.data import load_dataset

data_set_config = {
    "name": "chexpert",
    "root": f"{paths.data('chexpert')}",
    "patchsize": None,
    "resize_to": 384,
    "to_grayscale": True,
}
_, _, val_ds = load_dataset(**data_set_config)

In [ ]:
nn_indices_biomed = find_nearest_neighbor_batched_mm(inv_dict["original_biomed_embeddings"].to(device), val_ds, subject_model_biomed, batch_size=256)
nn_imgs_biomed = torch.stack([val_ds[ind.item()][0] for ind in nn_indices_biomed], dim=0)

In [ ]:
nn_indices_convnext = find_nearest_neighbor_batched_mm(inv_dict["original_convnext_embeddings"].to(device), val_ds, subject_model_convnext, batch_size=256)
nn_imgs_convnext = torch.stack([val_ds[ind.item()][0] for ind in nn_indices_convnext], dim=0)

In [ ]:
inv_dict["nn_imgs_convnext"] = nn_imgs_convnext
inv_dict["nn_imgs_biomed"] = nn_imgs_biomed
with torch.no_grad():
    convnext_nn_emb = []
    convnext_nn_cross_emb = []
    biomed_nn_emb = []
    biomed_nn_cross_emb = []
    for batch_conv, batch_bio in tqdm(zip(nn_imgs_convnext.split(100), nn_imgs_biomed.split(100))):
        convnext_nn_emb.append(subject_model_convnext(batch_conv.to(device)))
        convnext_nn_cross_emb.append(subject_model_biomed(batch_conv.to(device)))
        biomed_nn_emb.append(subject_model_biomed(batch_bio.to(device)))
        biomed_nn_cross_emb.append(subject_model_convnext(batch_bio.to(device)))
    inv_dict["nn_convnext_embeddings"] = torch.cat(convnext_nn_emb, dim=0)
    inv_dict["nn_biomed_embeddings"] = torch.cat(convnext_nn_cross_emb, dim=0) 
    inv_dict["nn_convnext_cross_embeddings"] = torch.cat(biomed_nn_emb, dim=0) 
    inv_dict["nn_biomed_cross_embeddings"] = torch.cat(biomed_nn_cross_emb, dim=0)
inv_dict["nn_indices_convnext"] = nn_indices_convnext
inv_dict["nn_indices_biomed"] = nn_indices_biomed

In [ ]:
torch.save(inv_dict, "sampled_chexpert_invariances.pt")

In [ ]:
with torch.no_grad():
    fiber_loss_biomed = ((inv_dict["original_biomed_embeddings"][:,None,...].softmax(dim=-1) - inv_dict["invariances_biomed_embeddings"].softmax(dim=-1)).abs()).sum(dim=-1)
    fiber_loss_convnext = ((inv_dict["original_convnext_embeddings"][:,None,...].softmax(dim=-1) - inv_dict["invariances_convnext_embeddings"].softmax(dim=-1)).abs()).sum(dim=-1)
    fiber_loss_biomed_nearest_neighbors = ((inv_dict["original_biomed_embeddings"].softmax(dim=-1) - inv_dict["nn_biomed_embeddings"].softmax(dim=-1).cpu() ).abs()).sum(dim=-1)
    fiber_loss_convnext_nearest_neighbors = ((inv_dict["original_convnext_embeddings"].softmax(dim=-1) - inv_dict["nn_convnext_embeddings"].softmax(dim=-1).cpu() ).abs()).sum(dim=-1)
    mean_fiber_loss_biomed_invariances = masked_mean(fiber_loss_biomed, inv_dict["masks"], 0)
    mean_fiber_loss_convnext_invariances = masked_mean(fiber_loss_convnext, inv_dict["masks"], 0)
    print(f"Fiber loss biomed invariances: {mean_fiber_loss_biomed_invariances.mean()} +- {mean_fiber_loss_biomed_invariances.std()}")
    print(f"Fiber loss convnext invariances: {mean_fiber_loss_convnext_invariances.mean()} +- {mean_fiber_loss_convnext_invariances.std()}")
    print(f"Fiber loss biomed nearest neighbors: {fiber_loss_biomed_nearest_neighbors.mean()}")
    print(f"Fiber loss convnext nearest neighbors: {fiber_loss_convnext_nearest_neighbors.mean()}")

In [ ]:
fiber_loss_biomed.quantile(0.95)

In [ ]:
with torch.no_grad():
    fiber_loss_biomed = ((inv_dict["original_biomed_embeddings"][:,None,...].softmax(dim=-1) - inv_dict["invariances_biomed_embeddings"].softmax(dim=-1)).abs()).mean(dim=-1)
    fiber_loss_convnext = ((inv_dict["original_convnext_embeddings"][:,None,...].softmax(dim=-1) - inv_dict["invariances_convnext_embeddings"].softmax(dim=-1)).abs()).mean(dim=-1)
    fiber_loss_biomed_nearest_neighbors = ((inv_dict["original_biomed_embeddings"].softmax(dim=-1) - inv_dict["nn_biomed_embeddings"].softmax(dim=-1).cpu() ).abs()).mean(dim=-1)
    fiber_loss_convnext_nearest_neighbors = ((inv_dict["original_convnext_embeddings"].softmax(dim=-1) - inv_dict["nn_convnext_embeddings"].softmax(dim=-1).cpu()).abs()).mean(dim=-1)
    mean_fiber_loss_biomed_invariances = masked_mean(fiber_loss_biomed, inv_dict["masks"], 0)
    mean_fiber_loss_convnext_invariances = masked_mean(fiber_loss_convnext, inv_dict["masks"], 0)
    print(f"Fiber loss biomed invariances: {mean_fiber_loss_biomed_invariances.mean()} +- {mean_fiber_loss_biomed_invariances.std()}")
    print(f"Fiber loss convnext invariances: {mean_fiber_loss_convnext_invariances.mean()} +- {mean_fiber_loss_convnext_invariances.std()}")
    print(f"Fiber loss biomed nearest neighbors: {fiber_loss_biomed_nearest_neighbors.mean()}")
    print(f"Fiber loss convnext nearest neighbors: {fiber_loss_convnext_nearest_neighbors.mean()}")

In [ ]:
plt_nn = True
num_samples = 5
samples_per_image = 2
fontsize = 24

plot_indices = torch.randperm(len(orig))[:num_samples]
print(plot_indices)
# plot_indices = [1287, 160, 224, 180, 114]
plots_per_row = 1 + samples_per_image*2 + plt_nn*2
plt.figure(figsize=(5*plots_per_row, 5*num_samples))


for i, ind in enumerate(plot_indices): 
    plt.subplot(num_samples, plots_per_row, i*plots_per_row+1)
    plt.imshow(orig[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5, cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("Originals", fontsize=fontsize)
    
    if plt_nn:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+2)
        plt.imshow(nn_imgs_biomed[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5, cmap="gray")
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$%%" % (fiber_loss_biomed_nearest_neighbors[ind]*100,), fontsize=fontsize)
        if i == 0:
            plt.title("Nearest Neighbor\nBiomedClip", fontsize=fontsize)

    for j in range(samples_per_image):
        plt.subplot(num_samples, plots_per_row, i*plots_per_row + 2 + plt_nn + j)
        fl_biomed_masked = fiber_loss_biomed[ind] + torch.where(mask[ind], 0.0, torch.inf)
        min_fl_ind = torch.argsort(fl_biomed_masked, dim=0, descending=False)[j]
        plt.imshow(inv_biomed[ind, min_fl_ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5, cmap="gray")
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$%%" % (fiber_loss_biomed[ind][min_fl_ind]*100, ), fontsize=fontsize)
        if i == 0:
            plt.title("BiomedClip\nInvariant Sample", fontsize=fontsize)
    
    if plt_nn:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+3+samples_per_image)
        plt.imshow(nn_imgs_convnext[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5, cmap="gray")
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$%%" % (fiber_loss_convnext_nearest_neighbors[ind]*100,), fontsize=fontsize)
        if i == 0:
            plt.title("Nearest Neighbor\nConvNeXt", fontsize=fontsize)
    
    for j in range(samples_per_image):
        plt.subplot(num_samples, plots_per_row, i*plots_per_row + 2 + plt_nn*2 + j + samples_per_image)
        fl_convnext_masked = fiber_loss_convnext[ind] + torch.where(mask[ind], 0.0, torch.inf)
        min_fl_ind = torch.argsort(fl_convnext_masked, dim=0, descending=False)[j]
        plt.imshow(inv_convnext[ind, min_fl_ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5, cmap="gray")
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$%%" % (fiber_loss_convnext[ind][min_fl_ind]*100, ), fontsize=fontsize)
        if i == 0:
            plt.title("ConvNeXt\nInvariant Sample", fontsize=fontsize)
plt.savefig(f"{paths.output('figures')}/RandomChexpertSamples.jpeg", bbox_inches="tight")
plt.show()

# Convnext vs Biomedclip larger head

In [ ]:
%cd sampled_invariances/

In [ ]:
!ls

In [ ]:
append_and_save_chexpert_samples(prefix_filter="sampled_train_invariances")

In [ ]:
invariance_dict = torch.load("appended_chexpert_invariances.pt")

In [ ]:
invariances_convnext = invariance_dict["invariances_convnext"]
invariances_biomed = invariance_dict["invariances_biomed"]
originals = invariance_dict["originals"]
labels = invariance_dict["labels"]
invariances_convnext_embeddings = invariance_dict["invariances_convnext_embeddings"]
invariances_biomed_embeddings = invariance_dict["invariances_biomed_embeddings"]
original_convnext_embeddings = invariance_dict["original_convnext_embeddings"]
original_biomed_embeddings = invariance_dict["original_biomed_embeddings"]
invariances_convnext_cross_embeddings = invariance_dict["invariances_convnext_cross_embeddings"]
invariances_biomed_cross_embeddings = invariance_dict["invariances_biomed_cross_embeddings"]

labels = labels[:,target_indices]

In [ ]:
original_convnext_embeddings_calibrated, scaler_convnext = compute_calibration(original_convnext_embeddings.to(device), labels.to(device))
original_biomed_embeddings_calibrated, scaler_biomed = compute_calibration(original_biomed_embeddings.to(device), labels.to(device))

threshold_convnext, f1_score_convnext = tune_threshold(original_convnext_embeddings_calibrated.to(device), labels.to(device))
threshold_biomed, f1_score_biomed = tune_threshold(original_biomed_embeddings_calibrated.to(device), labels.to(device))

In [ ]:
invariances_convnext_embeddings_calibrated = scaler_convnext(invariances_convnext_embeddings)
invariances_biomed_embeddings_calibrated = scaler_biomed(invariances_biomed_embeddings)
invariances_convnext_cross_embeddings_calibrated = scaler_biomed(invariances_convnext_cross_embeddings)
invariances_biomed_cross_embeddings_calibrated = scaler_convnext(invariances_biomed_cross_embeddings)

In [ ]:
plt.figure(figsize=(15, 25))
for i, ind in enumerate(torch.randperm(len(originals))[:5]):
    plt.subplot(5, 3, i*3+1)
    plt.imshow(originals[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
    plt.subplot(5, 3, i*3+2)
    plt.imshow(invariances_convnext[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
    plt.subplot(5, 3, i*3+3)
    plt.imshow(invariances_biomed[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
plt.show()

In [ ]:
difference_convnext_embeddings = invariances_convnext_embeddings_calibrated.sigmoid() - invariances_convnext_cross_embeddings_calibrated.sigmoid()
difference_biomed_embeddings = invariances_biomed_cross_embeddings_calibrated.sigmoid() - invariances_biomed_embeddings_calibrated.sigmoid()
plt.figure(figsize=(6*difference_convnext_embeddings.shape[1], 6))
plt.tight_layout()
for i in range(difference_convnext_embeddings.shape[1]):
    plt.subplot(1, difference_convnext_embeddings.shape[1], i+1)
    plt.title(f"Differences in probabilities for class {i}")
    plt.hist(difference_convnext_embeddings[:,i].cpu().detach().numpy(), range=[-0.5, 0.5], bins=20, density=False, alpha=0.5, label="ConvNext Invariances as Input")
    plt.hist(difference_biomed_embeddings[:,i].cpu().detach().numpy(), range=[-0.5, 0.5], bins=20, density=False, alpha=0.5, label="BiomedClip Invariances as Input")
    plt.ylabel(r"$p(\Delta)$", fontsize=14)
    plt.xlabel(r"$\Delta = p_\text{convnext} - p_\text{biomed}$", fontsize=14)
plt.legend()
plt.show()

In [ ]:
aurocs = {
    "convnext_originals": torch.zeros(5),
    "biomed_originals": torch.zeros(5),
    "convnext_relabeled": torch.zeros(5),
    "biomed_relabeled": torch.zeros(5),
    "convnext_cross_labeled": torch.zeros(5),
    "biomed_cross_labeled": torch.zeros(5),
    "convnext_same_labeled": torch.zeros(5),
    "biomed_same_labeled": torch.zeros(5),
    "convnext_cross_agreement": torch.zeros(5),
    "biomed_cross_agreement": torch.zeros(5),
}

In [ ]:
plt.figure(figsize=(6*original_convnext_embeddings_calibrated.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of Originals\n", fontsize=24)
plt.axis("off")



for i in range(original_convnext_embeddings_calibrated.shape[1]):
    plt.subplot(1, original_convnext_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(original_biomed_embeddings_calibrated[:,i], labels[:,i].bool(), num_bins=6)
    aurocs["biomed_originals"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="Biomed")
    fpr, tpr = compute_ROC_curve(original_convnext_embeddings_calibrated[:,i], labels[:,i].bool(), num_bins=6)
    aurocs["convnext_originals"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right")
    plt.title(target_columns[i])
plt.show()

In [ ]:
aurocs["biomed_originals"].mean()

In [ ]:
len(labels[:,2])

In [ ]:
originals_labeled_by_convnext = original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext
originals_labeled_by_biomed = original_biomed_embeddings_calibrated.sigmoid() > threshold_biomed
plt.figure(figsize=(6*original_convnext_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of cross labeled Originals\n", fontsize=24)
plt.axis("off")

for i in range(original_convnext_embeddings_calibrated.shape[1]):
    plt.subplot(1, original_convnext_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(original_biomed_embeddings_calibrated[:,i], originals_labeled_by_convnext[:,i], num_bins=6)
    aurocs["biomed_relabeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="Biomed using ConvNext as GT")
    fpr, tpr = compute_ROC_curve(original_convnext_embeddings_calibrated[:,i], originals_labeled_by_biomed[:,i], num_bins=6)
    aurocs["convnext_relabeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext using Biomed as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right")
    plt.title(target_columns[i])
plt.show()

In [ ]:
invariances_cross_labeled_by_convnext = invariances_biomed_cross_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_cross_labeled_by_biomed = invariances_convnext_cross_embeddings_calibrated.sigmoid() > threshold_biomed
plt.figure(figsize=(6*invariances_biomed_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of cross labeled Invariances\n", fontsize=24)
plt.axis("off")

for i in range(invariances_biomed_embeddings_calibrated.shape[1]):
    plt.subplot(1, invariances_biomed_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(invariances_biomed_embeddings_calibrated[:,i], invariances_cross_labeled_by_convnext[:,i])
    aurocs["biomed_cross_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="Biomed using ConvNext as GT")
    fpr, tpr = compute_ROC_curve(invariances_convnext_embeddings_calibrated[:,i], invariances_cross_labeled_by_biomed[:,i])
    aurocs["convnext_cross_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext using Biomed as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right")
    plt.title(target_columns[i])
plt.show()

In [ ]:
invariances_labeled_by_convnext = invariances_convnext_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_labeled_by_biomed = invariances_biomed_embeddings_calibrated.sigmoid() > threshold_biomed
plt.figure(figsize=(6*invariances_biomed_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of labeled cross Invariances\n", fontsize=24)
plt.axis("off")

for i in range(invariances_biomed_embeddings_calibrated.shape[1]):
    plt.subplot(1, invariances_biomed_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(invariances_convnext_cross_embeddings_calibrated[:,i], invariances_labeled_by_convnext[:,i])
    aurocs["biomed_same_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="Biomed using ConvNext as GT")
    fpr, tpr = compute_ROC_curve(invariances_biomed_cross_embeddings_calibrated[:,i], invariances_labeled_by_biomed[:,i])
    aurocs["convnext_same_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext using Biomed as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right")
    plt.title(target_columns[i])
plt.show()

In [ ]:
biomed_correct = (original_biomed_embeddings_calibrated.sigmoid() > threshold_biomed).to(device) == labels.to(device)
convnext_correct = (original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext).to(device) == labels.to(device)

both_models_correct = torch.logical_and(biomed_correct, convnext_correct)

invariances_cross_labeled_by_convnext = invariances_biomed_cross_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_cross_labeled_by_biomed = invariances_convnext_cross_embeddings_calibrated.sigmoid() > threshold_biomed


plt.figure(figsize=(6*invariances_biomed_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of cross labeled Invariances\n", fontsize=24)
plt.axis("off")

agreement_on_biomed_invariances = torch.zeros(5)
agreement_on_convnext_invariances = torch.zeros(5)
agreement_on_originals = ((original_biomed_embeddings_calibrated.sigmoid() > threshold_biomed) == (original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext)).float().mean(dim=0)


for i in range(invariances_biomed_embeddings_calibrated.shape[1]):
    plt.subplot(1, invariances_biomed_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(invariances_biomed_embeddings_calibrated[both_models_correct[:, i],i], invariances_cross_labeled_by_convnext[both_models_correct[:, i],i])
    aurocs["biomed_cross_agreement"][i] = torch.trapz(tpr, fpr)
    agreement_on_biomed_invariances[i] = (invariances_cross_labeled_by_convnext[both_models_correct[:, i],i] == labels.to(device)[both_models_correct[:, i],i]).float().mean()
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="Biomed using ConvNext as GT")    

    
    fpr, tpr = compute_ROC_curve(invariances_convnext_embeddings_calibrated[both_models_correct[:, i],i], invariances_cross_labeled_by_biomed[both_models_correct[:, i],i])
    aurocs["convnext_cross_agreement"][i] = torch.trapz(tpr, fpr)
    agreement_on_convnext_invariances[i] = (invariances_cross_labeled_by_biomed[both_models_correct[:, i],i] == labels.to(device)[both_models_correct[:, i],i]).float().mean()
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext using Biomed as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right")
    plt.title(target_columns[i])
plt.show()

In [ ]:
df = pd.DataFrame(aurocs).T
df.columns = [target_columns[i] for i in range(df.shape[1])]

df# .to_latex(float_format="{:.2f}".format, index=False,)

In [ ]:
print(agreement_on_biomed_invariances)
print(agreement_on_convnext_invariances)
print(agreement_on_originals)

print(agreement_on_biomed_invariances.mean())
print(agreement_on_convnext_invariances.mean())
print(agreement_on_originals.mean())

In [ ]:
agreement_marg_orig = agreement_on_originals.mean().item()
agreement_marg_convnext = agreement_on_convnext_invariances.mean().item()*100
agreement_marg_biomed = agreement_on_biomed_invariances.mean().item()*100

column_labels = ["ConvNeXt Invariant Set", "BiomedClip Invariant Set"]
x = np.arange(len(column_labels))
width = 1

plt.figure(figsize=(7, 4), dpi=200)

plt.bar(
    x,
    [agreement_marg_convnext, agreement_marg_biomed],
    width,
    color=["C0", "C1"],
    capsize=4,
)


plt.xticks(x, column_labels)
plt.ylabel("Classifier Agreement (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
column_labels = target_columns
x = np.arange(len(column_labels))
width = 0.25

plt.figure(figsize=(12, 4), dpi=200)

# plt.bar(
#     x - width,
#     agreement_on_originals.cpu().detach().numpy()*100,
#     width,
#     label="Original Samples",
#     capsize=4,
# )

plt.bar(
    x - width/2,
    agreement_on_convnext_invariances.cpu().detach().numpy()*100,
    width,
    label="ConvNeXt Invariant Set",
    capsize=4,
)

plt.bar(
    x + width/2,
    agreement_on_biomed_invariances.cpu().detach().numpy()*100,
    width,
    label="BiomedClip Invariant Set",
    capsize=4,
)


plt.xticks(x, column_labels)
plt.ylabel("Classifier Agreement (%)")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
contrastive_samples = {
    "originals": [],
    "samples": [],
    "labels": [],
    "embeddings_originals": [],
    "embeddings_samples": [],
}

for class_ind in range(5):
    not_disagree_on_orig = originals_labeled_by_biomed[:,class_ind] == originals_labeled_by_convnext[:,class_ind]
    not_disagree_on_orig = torch.logical_and(originals_labeled_by_biomed[:,class_ind] == labels[:,class_ind].to(device), not_disagree_on_orig)
    disagree_on_fiber = invariances_labeled_by_biomed[:,class_ind] != invariances_cross_labeled_by_convnext[:,class_ind]
    disagree_on_fiber = torch.logical_and(disagree_on_fiber, not_disagree_on_orig)
    print(sum(disagree_on_fiber))
    
    contrastive_samples["originals"].append(originals[disagree_on_fiber.cpu()])
    contrastive_samples["samples"].append(invariances_biomed[disagree_on_fiber.cpu()])
    contrastive_samples["labels"].append(torch.ones((disagree_on_fiber.sum().item(),), device="cpu", dtype=torch.long)*class_ind)
    contrastive_samples["embeddings_originals"].append(original_biomed_embeddings[disagree_on_fiber.cpu()])
    contrastive_samples["embeddings_samples"].append(invariances_biomed_embeddings[disagree_on_fiber.cpu()])

contrastive_samples["originals"] = torch.cat(contrastive_samples["originals"], dim=0)
contrastive_samples["samples"] = torch.cat(contrastive_samples["samples"], dim=0)
contrastive_samples["labels"] = torch.cat(contrastive_samples["labels"], dim=0)
contrastive_samples["embeddings_originals"] = torch.cat(contrastive_samples["embeddings_originals"], dim=0)
contrastive_samples["embeddings_samples"] = torch.cat(contrastive_samples["embeddings_samples"], dim=0)

torch.save(contrastive_samples, "../contrastive_samples.pt")


In [ ]:
print(originals.shape)

In [ ]:
invariances_cross_labeled_by_convnext = invariances_biomed_cross_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_labeled_by_biomed = invariances_biomed_embeddings_calibrated.sigmoid() > threshold_biomed
original_labeled_by_biomed = original_biomed_embeddings_calibrated.sigmoid() > threshold_biomed
biomed_correct = original_labeled_by_biomed.to(device) == labels.to(device)
class_ind = 1
no_disease_seen = torch.logical_and(biomed_correct[:,class_ind], labels.to(device)[:,class_ind]==0)
disease_overlooked = torch.logical_and(no_disease_seen, invariances_cross_labeled_by_convnext[:,class_ind])
disease_overlooked = torch.logical_and(disease_overlooked, invariances_labeled_by_biomed[:,class_ind]==0)
disease_overlooked = torch.logical_and(disease_overlooked, original_convnext_embeddings_calibrated.sigmoid()[:,class_ind]<0.25)
print(sum(disease_overlooked))

In [ ]:
len(originals)

In [ ]:
original_biomed_embeddings_calibrated[:,1].sigmoid().mean()

In [ ]:
ind_probs_sorted = torch.argsort(invariances_biomed_cross_embeddings_calibrated.sigmoid()[disease_overlooked, class_ind])
print(invariances_biomed_cross_embeddings_calibrated.sigmoid()[disease_overlooked, class_ind][ind_probs_sorted[-10:]])

In [ ]:
# ind_disease_overlooked = torch.where(disease_overlooked)[0]
plt.figure(figsize=(10, 25))
for i, ind in enumerate(ind_probs_sorted[-3:]):
    plt.subplot(5, 2, i*2+1)
    if i == 0:
        plt.title("Original", fontsize=18)
    p_original_biomed = original_biomed_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    p_original_convnext = original_convnext_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    plt.imshow(originals[disease_overlooked.cpu()][ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(
        rf"BiomedClip: $p(\text{{Cardiomegaly}}) = {p_original_biomed.item():.1f}\%$"
        "\n"
        rf"ConvNeXt: $p(\text{{Cardiomegaly}}) = {p_original_convnext.item():.1f}\%$",
        fontsize=14
    )

    plt.subplot(5, 2, i*2+2)
    if i == 0:
        plt.title("BiomedClip\nInvariant Sample", fontsize=18)
    p_invariances_biomed = invariances_biomed_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    p_invariances_convnext = invariances_biomed_cross_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    plt.imshow(invariances_biomed[disease_overlooked.cpu()][ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(
        rf"BiomedClip: $p(\text{{Cardiomegaly}}) = {p_invariances_biomed.item():.1f}\%$"
        "\n"
        rf"ConvNeXt: $p(\text{{Cardiomegaly}}) = {p_invariances_convnext.item():.1f}\%$",
        fontsize=14
    )
plt.savefig(f"{paths.output('figures')}/CardiomegalyBiomed.jpeg", bbox_inches="tight")
plt.show()

In [ ]:
original_biomed_embeddings_calibrated.sigmoid()[disease_overlooked].shape

In [ ]:
invariances_cross_labeled_by_biomed = invariances_convnext_cross_embeddings_calibrated.sigmoid() > threshold_biomed
invariances_labeled_by_convnext = invariances_convnext_embeddings_calibrated.sigmoid() > threshold_convnext
original_labeled_by_convnext = original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext
convnext_correct = original_labeled_by_convnext == labels.to(device)
class_ind = 1
no_disease_seen = torch.logical_and(convnext_correct[:,class_ind], labels.to(device)[:,class_ind]==0)
disease_overlooked = torch.logical_and(no_disease_seen, invariances_cross_labeled_by_biomed[:,class_ind])
disease_overlooked = torch.logical_and(disease_overlooked, invariances_labeled_by_convnext[:,class_ind]==0)
disease_overlooked = torch.logical_and(disease_overlooked, original_biomed_embeddings_calibrated.sigmoid()[:,class_ind]<0.25)
print(sum(disease_overlooked))

In [ ]:
ind_probs_sorted = torch.argsort(invariances_convnext_cross_embeddings_calibrated.sigmoid()[disease_overlooked, class_ind])
print(invariances_convnext_cross_embeddings_calibrated.sigmoid()[disease_overlooked, class_ind][ind_probs_sorted[-10:]])

In [ ]:
inds_selected = ind_probs_sorted[-3], ind_probs_sorted[-4], ind_probs_sorted[-1]

In [ ]:
# ind_disease_overlooked = torch.where(disease_overlooked)[0]
plt.figure(figsize=(10, 25))
for i, ind in enumerate(inds_selected):
    plt.subplot(5, 2, i*2+1)
    if i == 0:
        plt.title("Original", fontsize=18)
    p_original_biomed = original_biomed_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    p_original_convnext = original_convnext_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    plt.imshow(originals[disease_overlooked.cpu()][ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(
        rf"ConvNeXt: $p(\text{{Cardiomegaly}}) = {p_original_convnext.item():.1f}\%$"
        "\n"
        rf"BiomedClip: $p(\text{{Cardiomegaly}}) = {p_original_biomed.item():.1f}\%$",
        fontsize=14
    )

    plt.subplot(5, 2, i*2+2)
    if i == 0:
        plt.title("ConvNeXt\nInvariant Sample", fontsize=18)
    p_invariances_biomed = invariances_convnext_cross_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    p_invariances_convnext = invariances_convnext_embeddings_calibrated.sigmoid()[disease_overlooked][ind, class_ind]*100
    plt.imshow(invariances_biomed[disease_overlooked.cpu()][ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(
        rf"ConvNeXt: $p(\text{{Cardiomegaly}}) = {p_invariances_convnext.item():.1f}\%$"
        "\n"
        rf"BiomedClip: $p(\text{{Cardiomegaly}}) = {p_invariances_biomed.item():.1f}\%$",
        fontsize=14
    )
plt.savefig(f"{paths.output('figures')}/CardiomegalyConvnext.jpeg", bbox_inches="tight")
plt.show()

# Convnext vs. Convnext

In [ ]:
def append_and_save_chexpert_samples(prefix_filter="sampled_invariances_", save_name="appended_chexpert_invariances.pt"):
    combined_dict = {
        "invariances_convnext": [],
        "invariances_convnext_2": [],
        "originals": [],
        "labels": [],
        "invariances_convnext_embeddings": [],
        "invariances_convnext_2_embeddings": [],
        "original_convnext_embeddings": [],
        "original_convnext_2_embeddings": [],
        "invariances_convnext_cross_embeddings": [],
        "invariances_convnext_2_cross_embeddings": [],
    }
    
    for fname in tqdm(os.listdir(".")):
        if fname.startswith(prefix_filter):
            results_dict = torch.load(fname)
            for key in results_dict.keys():
                combined_dict[key].append(results_dict[key])
    for key in combined_dict.keys():
        combined_dict[key] = torch.cat(combined_dict[key], dim=0)
    torch.save(combined_dict, save_name)

In [ ]:
def combine_and_save_chexpert_samples(prefix_filter="sampled_invariances_", save_name="sampled_chexpert_invariances.pt"):
    combined_dict = {
        "invariances_convnext": [],
        "invariances_convnext_2": [],
        "originals": [],
        "labels": [],
        "invariances_convnext_embeddings": [],
        "invariances_convnext_2_embeddings": [],
        "original_convnext_embeddings": [],
        "original_convnext_2_embeddings": [],
        "invariances_convnext_cross_embeddings": [],
        "invariances_convnext_2_cross_embeddings": [],
    }
    
    for fname in tqdm(os.listdir(".")):
        if fname.startswith(prefix_filter):
            results_dict = torch.load(fname)
            for key in results_dict.keys():
                combined_dict[key].append(results_dict[key])
    
    # Check consistency:
    for last_orig, next_orig in zip(combined_dict["originals"][:-1], combined_dict["originals"][1:]):
        min_len = min(len(last_orig), len(next_orig))
        assert torch.all(((last_orig[:min_len] - next_orig[:min_len])**2).sum(dim=-1) < 1.e-4), "Dataset order does not agree"
    
    max_len, ind_max = 0, None
    for i, orig in enumerate(combined_dict["originals"]):
        if len(orig) > max_len:
            ind_max = i
            max_len = len(orig)
    
    combined_dict["originals"] = combined_dict["originals"][ind_max]
    combined_dict["original_convnext_embeddings"] = combined_dict["original_convnext_embeddings"][ind_max]
    combined_dict["original_convnext_2_embeddings"] = combined_dict["original_convnext_2_embeddings"][ind_max]
    combined_dict["labels"] = combined_dict["labels"][ind_max].to(device)
    
    assert len(combined_dict["originals"]) == len(combined_dict["original_convnext_embeddings"]) == len(combined_dict["original_convnext_2_embeddings"])
    assert len(combined_dict["originals"]) == len(combined_dict["labels"])
    
    combined_dict["masks"] = []
    for i, (inv_convnext, 
            inv_convnext_embeddings, 
            inv_convnext_cross_embeddings, 
            inv_convnext_2, 
            inv_convnext_2_embeddings, 
            inv_convnext_2_cross_embeddings) in enumerate(zip(combined_dict["invariances_convnext"], 
                                                           combined_dict["invariances_convnext_embeddings"],
                                                           combined_dict["invariances_convnext_cross_embeddings"],
                                                           combined_dict["invariances_convnext_2"],
                                                           combined_dict["invariances_convnext_2_embeddings"],
                                                           combined_dict["invariances_convnext_2_cross_embeddings"])):
        
        assert len(inv_convnext) == len(inv_convnext_embeddings) == len(inv_convnext_2) == len(inv_convnext_2_embeddings) == len(inv_convnext_cross_embeddings) == len(inv_convnext_2_cross_embeddings)
        mask = torch.cat((torch.ones(len(inv_convnext), device=inv_convnext.device, dtype=bool), 
                          torch.zeros(max_len - len(inv_convnext), device=inv_convnext.device, dtype=bool)), dim=0)
        inv_convnext = torch.cat((inv_convnext, torch.zeros(max_len - len(inv_convnext), *inv_convnext.shape[1:], device=inv_convnext.device, dtype=inv_convnext.dtype)), dim=0)
        inv_convnext_embeddings = torch.cat((inv_convnext_embeddings, torch.zeros(max_len - len(inv_convnext_embeddings), *inv_convnext_embeddings.shape[1:], device=inv_convnext_embeddings.device, dtype=inv_convnext_embeddings.dtype)), dim=0)
        inv_convnext_cross_embeddings = torch.cat((inv_convnext_cross_embeddings, torch.zeros(max_len - len(inv_convnext_cross_embeddings), *inv_convnext_cross_embeddings.shape[1:], device=inv_convnext_cross_embeddings.device, dtype=inv_convnext_cross_embeddings.dtype)), dim=0)
        inv_convnext_2 = torch.cat((inv_convnext_2, torch.zeros(max_len - len(inv_convnext_2), *inv_convnext_2.shape[1:], device=inv_convnext_2.device, dtype=inv_convnext_2.dtype)), dim=0)
        inv_convnext_2_embeddings = torch.cat((inv_convnext_2_embeddings, torch.zeros(max_len - len(inv_convnext_2_embeddings), *inv_convnext_2_embeddings.shape[1:], device=inv_convnext_2_embeddings.device, dtype=inv_convnext_2_embeddings.dtype)), dim=0)
        inv_convnext_2_cross_embeddings = torch.cat((inv_convnext_2_cross_embeddings, torch.zeros(max_len - len(inv_convnext_2_cross_embeddings), *inv_convnext_2_cross_embeddings.shape[1:], device=inv_convnext_2_cross_embeddings.device, dtype=inv_convnext_2_cross_embeddings.dtype)), dim=0)
        combined_dict["invariances_convnext"][i] = inv_convnext
        combined_dict["invariances_convnext_embeddings"][i] = inv_convnext_embeddings
        combined_dict["invariances_convnext_cross_embeddings"][i] = inv_convnext_cross_embeddings
        combined_dict["invariances_convnext_2"][i] = inv_convnext_2
        combined_dict["invariances_convnext_2_embeddings"][i] = inv_convnext_2_embeddings
        combined_dict["invariances_convnext_2_cross_embeddings"][i] = inv_convnext_2_cross_embeddings
        combined_dict["masks"].append(mask)
    
    combined_dict["invariances_convnext"] = torch.stack(combined_dict["invariances_convnext"], dim=1)
    combined_dict["invariances_convnext_embeddings"] = torch.stack(combined_dict["invariances_convnext_embeddings"], dim=1)
    combined_dict["invariances_convnext_cross_embeddings"] = torch.stack(combined_dict["invariances_convnext_cross_embeddings"], dim=1)
    combined_dict["invariances_convnext_2"] = torch.stack(combined_dict["invariances_convnext_2"], dim=1)
    combined_dict["invariances_convnext_2_embeddings"] = torch.stack(combined_dict["invariances_convnext_2_embeddings"], dim=1)
    combined_dict["invariances_convnext_2_cross_embeddings"] = torch.stack(combined_dict["invariances_convnext_2_cross_embeddings"], dim=1)
    combined_dict["masks"] = torch.stack(combined_dict["masks"], dim=1)
    
    torch.save(combined_dict, save_name)

In [ ]:
os.chdir(paths.output("chexpert", "sampled_invariances", "convnext_only/v6"))

In [ ]:
append_and_save_chexpert_samples()

In [ ]:
invariance_dict = torch.load(f"{paths.output('chexpert', 'sampled_invariances')}/convnext_only/v6/appended_chexpert_invariances.pt")
# invariance_dict = torch.load(f"{paths.data()}/sampled_invariances_15_08_59__23_01_2026_41905.pt")

In [ ]:
invariances_convnext = invariance_dict["invariances_convnext"]
invariances_convnext_2 = invariance_dict["invariances_convnext_2"]
originals = invariance_dict["originals"]
labels = invariance_dict["labels"]
invariances_convnext_embeddings = invariance_dict["invariances_convnext_embeddings"]
invariances_convnext_2_embeddings = invariance_dict["invariances_convnext_2_embeddings"]
original_convnext_embeddings = invariance_dict["original_convnext_embeddings"]
original_convnext_2_embeddings = invariance_dict["original_convnext_2_embeddings"]
invariances_convnext_cross_embeddings = invariance_dict["invariances_convnext_cross_embeddings"]
invariances_convnext_2_cross_embeddings = invariance_dict["invariances_convnext_2_cross_embeddings"]

labels = labels[:,target_indices]

In [ ]:
original_convnext_embeddings_calibrated, scaler_convnext = compute_calibration(original_convnext_embeddings.to(device), labels.to(device))
original_convnext_2_embeddings_calibrated, scaler_convnext_2 = compute_calibration(original_convnext_2_embeddings.to(device), labels.to(device))

threshold_convnext, f1_score_convnext = tune_threshold(original_convnext_embeddings_calibrated.to(device), labels.to(device))
threshold_convnext_2, f1_score_convnext_2 = tune_threshold(original_convnext_2_embeddings_calibrated.to(device), labels.to(device))

In [ ]:
invariances_convnext_embeddings_calibrated = scaler_convnext(invariances_convnext_embeddings)
invariances_convnext_2_embeddings_calibrated = scaler_convnext_2(invariances_convnext_2_embeddings)
invariances_convnext_cross_embeddings_calibrated = scaler_convnext_2(invariances_convnext_cross_embeddings)
invariances_convnext_2_cross_embeddings_calibrated = scaler_convnext(invariances_convnext_2_cross_embeddings)

In [ ]:
indices = torch.randperm(len(originals))[:5]
print(indices)
# indices = [3614, 108, 1505, 1439]


plt.figure(figsize=(15, 25))
for i, ind in enumerate(indices):
    plt.subplot(5, 3, i*3+1)
    plt.imshow(originals[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
    plt.subplot(5, 3, i*3+2)
    plt.imshow(invariances_convnext[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
    plt.subplot(5, 3, i*3+3)
    plt.imshow(invariances_convnext_2[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
plt.show()

In [ ]:
difference_convnext_embeddings = invariances_convnext_embeddings_calibrated.sigmoid() - invariances_convnext_cross_embeddings_calibrated.sigmoid()
difference_convnext_2_embeddings = invariances_convnext_2_cross_embeddings_calibrated.sigmoid() - invariances_convnext_2_embeddings_calibrated.sigmoid()
plt.figure(figsize=(6*difference_convnext_embeddings.shape[1], 6))
plt.tight_layout()
for i in range(difference_convnext_embeddings.shape[1]):
    plt.subplot(1, difference_convnext_embeddings.shape[1], i+1)
    plt.title(f"Differences in probabilities for class {i}")
    plt.hist(difference_convnext_embeddings[:,i].cpu().detach().numpy(), range=[-0.5, 0.5], bins=20, density=False, alpha=0.5, label="ConvNext Invariances as Input")
    plt.hist(difference_convnext_2_embeddings[:,i].cpu().detach().numpy(), range=[-0.5, 0.5], bins=20, density=False, alpha=0.5, label="ConvNeXt BClip Invariances as Input")
    plt.ylabel(r"$p(\Delta)$", fontsize=14)
    plt.xlabel(r"$\Delta = p_\text{ConvNeXt A} - p_\text{ConvNeXt B}$", fontsize=14)
plt.legend()
plt.show()

In [ ]:
aurocs = {
    "convnext_originals": torch.zeros(5),
    "convnext_2_originals": torch.zeros(5),
    "convnext_relabeled": torch.zeros(5),
    "convnext_2_relabeled": torch.zeros(5),
    "convnext_cross_labeled": torch.zeros(5),
    "convnext_2_cross_labeled": torch.zeros(5),
    "convnext_same_labeled": torch.zeros(5),
    "convnext_2_same_labeled": torch.zeros(5),
    "convnext_cross_agreement": torch.zeros(5),
    "convnext_2_cross_agreement": torch.zeros(5),
}

In [ ]:
plt.figure(figsize=(6*original_convnext_embeddings_calibrated.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of Originals\n", fontsize=24)
plt.axis("off")

for i in range(original_convnext_embeddings_calibrated.shape[1]):
    plt.subplot(1, original_convnext_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(original_convnext_2_embeddings_calibrated[:,i], labels[:,i].bool(), num_bins=5)
    aurocs["convnext_originals"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNeXt B")
    fpr, tpr = compute_ROC_curve(original_convnext_embeddings_calibrated[:,i], labels[:,i].bool(), num_bins=5)
    aurocs["convnext_2_originals"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext A")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right")
    plt.title(target_columns[i])
plt.show()

In [ ]:
originals_labeled_by_convnext = original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext
originals_labeled_by_convnext_2 = original_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext_2
plt.figure(figsize=(6*original_convnext_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of cross labeled Originals\n", fontsize=24)
plt.axis("off")

for i in range(original_convnext_embeddings_calibrated.shape[1]):
    plt.subplot(1, original_convnext_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(original_convnext_2_embeddings_calibrated[:,i], originals_labeled_by_convnext[:,i])
    aurocs["convnext_relabeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNeXt B using ConvNext A as GT")
    fpr, tpr = compute_ROC_curve(original_convnext_embeddings_calibrated[:,i], originals_labeled_by_convnext_2[:,i])
    aurocs["convnext_2_relabeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext A using ConvNeXt B as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right", fontsize=12)
    plt.title(target_columns[i])
plt.show()

In [ ]:
invariances_cross_labeled_by_convnext = invariances_convnext_2_cross_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_cross_labeled_by_convnext_2 = invariances_convnext_cross_embeddings_calibrated.sigmoid() > threshold_convnext_2
plt.figure(figsize=(6*invariances_convnext_2_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of cross labeled Invariances\n", fontsize=24)
plt.axis("off")

for i in range(invariances_convnext_2_embeddings_calibrated.shape[1]):
    plt.subplot(1, invariances_convnext_2_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(invariances_convnext_2_embeddings_calibrated[:,i], invariances_cross_labeled_by_convnext[:,i])
    aurocs["convnext_cross_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNeXt B using ConvNext A as GT")
    fpr, tpr = compute_ROC_curve(invariances_convnext_embeddings_calibrated[:,i], invariances_cross_labeled_by_convnext_2[:,i])
    aurocs["convnext_2_cross_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext A using ConvNeXt B as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right", fontsize=12)
    plt.title(target_columns[i])
plt.show()

In [ ]:
invariances_labeled_by_convnext = invariances_convnext_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_labeled_by_convnext_2 = invariances_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext_2
plt.figure(figsize=(6*invariances_convnext_2_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of labeled cross Invariances\n", fontsize=24)
plt.axis("off")

for i in range(invariances_convnext_2_embeddings_calibrated.shape[1]):
    plt.subplot(1, invariances_convnext_2_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(invariances_convnext_cross_embeddings_calibrated[:,i], invariances_labeled_by_convnext[:,i])
    aurocs["convnext_same_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNeXt B using ConvNext A as GT")
    fpr, tpr = compute_ROC_curve(invariances_convnext_2_cross_embeddings_calibrated[:,i], invariances_labeled_by_convnext_2[:,i])
    aurocs["convnext_2_same_labeled"][i] = torch.trapz(tpr, fpr)
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext A using ConvNeXt B as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right", fontsize=12)
    plt.title(target_columns[i])
plt.show()

In [ ]:

convnext_2_correct = (original_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext_2).to(device) == labels.to(device)
convnext_correct = (original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext).to(device) == labels.to(device)

both_models_correct = torch.logical_and(convnext_2_correct, convnext_correct)

invariances_cross_labeled_by_convnext = invariances_convnext_2_cross_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_cross_labeled_by_convnext_2 = invariances_convnext_cross_embeddings_calibrated.sigmoid() > threshold_convnext_2


plt.figure(figsize=(6*invariances_convnext_2_embeddings.shape[1], 6))
plt.tight_layout()
plt.title("ROC Curves of cross labeled Invariances\n", fontsize=24)
plt.axis("off")

agreement_on_convnext_2_invariances = torch.zeros(5)
agreement_on_convnext_invariances = torch.zeros(5)
agreement_on_originals = ((original_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext_2) == (original_convnext_embeddings_calibrated.sigmoid() > threshold_convnext)).float().mean(dim=0)


for i in range(invariances_convnext_2_embeddings_calibrated.shape[1]):
    plt.subplot(1, invariances_convnext_2_embeddings_calibrated.shape[1], i+1)
    fpr, tpr = compute_ROC_curve(invariances_convnext_2_embeddings_calibrated[both_models_correct[:, i],i], invariances_cross_labeled_by_convnext[both_models_correct[:, i],i])
    aurocs["convnext_2_cross_agreement"][i] = torch.trapz(tpr, fpr)
    agreement_on_convnext_2_invariances[i] = (invariances_cross_labeled_by_convnext[both_models_correct[:, i],i] == labels.to(device)[both_models_correct[:, i],i]).float().mean()
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNeXt B using ConvNext A as GT")    

    
    fpr, tpr = compute_ROC_curve(invariances_convnext_embeddings_calibrated[both_models_correct[:, i],i], invariances_cross_labeled_by_convnext_2[both_models_correct[:, i],i])
    aurocs["convnext_cross_agreement"][i] = torch.trapz(tpr, fpr)
    agreement_on_convnext_invariances[i] = (invariances_cross_labeled_by_convnext_2[both_models_correct[:, i],i] == labels.to(device)[both_models_correct[:, i],i]).float().mean()
    plt.plot(fpr.cpu().detach().numpy(), tpr.cpu().detach().numpy(), label="ConvNext A using ConvNeXt B as GT")
    plt.plot([0, 1], [0, 1], color="black", linestyle="--")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")
    plt.legend(loc="lower right", fontsize=12)
    plt.title(target_columns[i])
plt.show()

In [ ]:
invariances_cross_labeled_by_convnext = invariances_convnext_2_cross_embeddings_calibrated.sigmoid() > threshold_convnext
invariances_labeled_by_convnext_2 = invariances_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext_2
original_labeled_by_convnext_2 = original_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext
convnext_2_correct = (original_convnext_2_embeddings_calibrated.sigmoid() > threshold_convnext).to(device) == labels.to(device)
class_ind = 4
no_disease_seen = torch.logical_and(convnext_2_correct[:,class_ind], labels.to(device)[:,class_ind]==0)
disease_overlooked = torch.logical_and(no_disease_seen, invariances_cross_labeled_by_convnext[:,class_ind])
disease_overlooked = torch.logical_and(disease_overlooked, invariances_labeled_by_convnext_2[:,class_ind]==0)
print(sum(disease_overlooked))

In [ ]:
ind_disease_overlooked = torch.where(disease_overlooked)[0]
plt.figure(figsize=(10, 25))
for i, ind in enumerate(ind_disease_overlooked[torch.randperm(len(ind_disease_overlooked))[:5]]):
    plt.subplot(5, 2, i*2+1)
    plt.imshow(originals[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
    plt.subplot(5, 2, i*2+2)
    plt.imshow(invariances_convnext_2[ind].squeeze().cpu().detach().numpy(), cmap="gray")
    plt.axis("off")
plt.show()

In [ ]:
print(agreement_on_convnext_2_invariances)
print(agreement_on_convnext_invariances)
print(agreement_on_originals)

print(agreement_on_convnext_2_invariances.mean())
print(agreement_on_convnext_invariances.mean())
print(agreement_on_originals.mean())

In [ ]:
agreement_marg_orig = agreement_on_originals.mean().item()
agreement_marg_convnext = agreement_on_convnext_invariances.mean().item()*100
agreement_marg_convnext_2 = agreement_on_convnext_2_invariances.mean().item()*100

column_labels = ["ConvNeXt A Invariant Set", "ConvNeXt B Invariant Set"]
x = np.arange(len(column_labels))
width = 1

plt.figure(figsize=(7, 4), dpi=200)

plt.bar(
    x,
    [agreement_marg_convnext, agreement_marg_convnext_2],
    width,
    color=["C0", "C1"],
    capsize=4,
)


plt.xticks(x, column_labels)
plt.ylabel("Classifier Agreement (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
column_labels = target_columns
x = np.arange(len(column_labels))
width = 0.25

plt.figure(figsize=(12, 4), dpi=200)

# plt.bar(
#     x - width,
#     agreement_on_originals.cpu().detach().numpy()*100,
#     width,
#     label="Original Samples",
#     capsize=4,
# )

plt.bar(
    x - width/2,
    agreement_on_convnext_invariances.cpu().detach().numpy()*100,
    width,
    label="ConvNeXt A Invariant Set",
    capsize=4,
)

plt.bar(
    x + width/2,
    agreement_on_convnext_2_invariances.cpu().detach().numpy()*100,
    width,
    label="ConvNeXt B Invariant Set",
    capsize=4,
)


plt.xticks(x, column_labels)
plt.ylabel("Classifier Agreement (%)")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()